In [1]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
REPLAY_JSON_PATH = PROJECT_ROOT / "replay_parser" / "output" / "parsed_match.json"

with open(REPLAY_JSON_PATH, "r") as f:
    replay_data = json.load(f)

print("Top-level keys:", list(replay_data.keys()))
for key, value in replay_data.items():
    print(f"{key}: {len(value)} entries")

Top-level keys: ['players', 'deaths', 'structures_destroyed', 'patron_events', 'mid_boss_kills', 'rift_captures', 'urn_pickups', 'stat_snapshots']
players: 12 entries
deaths: 120 entries
structures_destroyed: 19 entries
patron_events: 1 entries
mid_boss_kills: 3 entries
rift_captures: 6 entries
urn_pickups: 10 entries
stat_snapshots: 205 entries


In [4]:
# team for players
players_df = pd.DataFrame(replay_data["players"])
controller_to_team = dict(zip(players_df["controller_index"], players_df["team"]))
print(players_df)

# long format DataFrame
rows = []
for snapshot in replay_data["stat_snapshots"]:
    t = snapshot["game_time_s"]
    for p in snapshot["players"]:
        rows.append({
            "game_time_s": t,
            "controller_index": p["controller_index"],
            "team": p["team"],
            "kills": p["kills"],
            "deaths": p["deaths"],
            "assists": p["assists"],
            "hero_damage": p["hero_damage"],
            "objective_damage": p["objective_damage"],
            "gold_net_worth": p["gold_net_worth"],
        })

snapshots_long = pd.DataFrame(rows)
print(snapshots_long.shape)
snapshots_long.head(10)

    controller_index  pawn_index  team            username
0                  2        3429     2             seiryuu
1                  3        3290     2             Numlegs
2                  4        3568     2  Demui on Twitch :D
3                  5        3495     2                Tech
4                  6          77     3               Docha
5                  7          84     3              Specty
6                  8        3475     2            Seikatsu
7                  9          97     3                Anri
8                 10          98     3               Robin
9                 11          94     3     Little Boy 1945
10                12          89     2              Golova
11                13        3534     3          privativev
(2435, 9)


,game_time_s,controller_index,team,kills,deaths,assists,hero_damage,objective_damage,gold_net_worth
0,15.03125,5,2,0,0,0,0,0,600
1,15.03125,6,3,0,0,0,0,0,600
2,15.03125,10,3,0,0,0,0,0,600
3,15.03125,11,3,0,0,0,0,0,600
4,15.03125,13,3,0,0,0,0,0,600
5,30.03125,5,2,0,0,0,0,0,600
6,30.03125,6,3,0,0,0,0,0,600
7,30.03125,10,3,0,0,0,0,0,600
8,30.03125,11,3,0,0,0,0,0,600
9,30.03125,12,2,0,0,0,0,0,600


In [6]:
team_snapshots = (
    snapshots_long
    .groupby(["game_time_s", "team"], as_index=False)
    .agg({
        "kills": "sum",
        "deaths": "sum",
        "assists": "sum",
        "hero_damage": "sum",
        "objective_damage": "sum",
        "gold_net_worth": "sum",
    })
)

print(team_snapshots.shape)
print(team_snapshots.head(20))

(408, 8)
    game_time_s  team  kills  deaths  assists  hero_damage  objective_damage  \
0      15.03125     2      0       0        0            0                 0   
1      15.03125     3      0       0        0            0                 0   
2      30.03125     2      0       0        0            0                 0   
3      30.03125     3      0       0        0            0                 0   
4      45.03125     2      0       0        0            0                 0   
5      45.03125     3      0       0        0            0                 0   
6      60.03125     2      0       0        0          213                 0   
7      60.03125     3      0       0        0          300                 0   
8      75.03125     2      0       0        0          887                 0   
9      75.03125     3      0       0        0          640                 0   
10     90.03125     2      0       0        0         1071                 0   
11     90.03125     3      0   

In [10]:
print(team_snapshots.tail(10))

     game_time_s  team  kills  deaths  assists  hero_damage  objective_damage  \
398  3000.046875     2     55      55      110       292164             21858   
399  3000.046875     3     55      55      106       264389             60243   
400  3015.046875     2     55      55      110       296633             21858   
401  3015.046875     3     55      55      106       267175             64569   
402  3030.046875     2     56      57      111       303760             21858   
403  3030.046875     3     57      56      116       277521             67049   
404  3045.046875     2     56      57      111       303760             21858   
405  3045.046875     3     57      56      116       277521             67049   
406  3060.046875     2     56      57      111       303760             21858   
407  3060.046875     3     57      56      116       277521             67049   

     gold_net_worth  
398          315532  
399          376216  
400          315661  
401          389154 

In [12]:
team_wide = (
    team_snapshots
    .pivot(index="game_time_s", columns="team")
)

# Flatten the multi-level column names
team_wide.columns = [
    f"{stat}_team{team}"
    for stat, team in team_wide.columns
]

team_wide = team_wide.reset_index()

print(team_wide.shape)
print(team_wide.head(10))

(204, 13)
   game_time_s  kills_team2  kills_team3  deaths_team2  deaths_team3  \
0     15.03125            0            0             0             0   
1     30.03125            0            0             0             0   
2     45.03125            0            0             0             0   
3     60.03125            0            0             0             0   
4     75.03125            0            0             0             0   
5     90.03125            0            0             0             0   
6    105.03125            0            0             0             0   
7    120.03125            0            0             0             0   
8    135.03125            1            0             0             1   
9    150.03125            1            0             0             1   

   assists_team2  assists_team3  hero_damage_team2  hero_damage_team3  \
0              0              0                  0                  0   
1              0              0                  0 

In [14]:
diff_df = pd.DataFrame({
    "game_time_s": team_wide["game_time_s"],

    "kills_diff": (
        team_wide["kills_team2"] -
        team_wide["kills_team3"]
    ),

    "deaths_diff": (
        team_wide["deaths_team2"] -
        team_wide["deaths_team3"]
    ),

    "assists_diff": (
        team_wide["assists_team2"] -
        team_wide["assists_team3"]
    ),

    "hero_damage_diff": (
        team_wide["hero_damage_team2"] -
        team_wide["hero_damage_team3"]
    ),

    "objective_damage_diff": (
        team_wide["objective_damage_team2"] -
        team_wide["objective_damage_team3"]
    ),

    "net_worth_diff": (
        team_wide["gold_net_worth_team2"] -
        team_wide["gold_net_worth_team3"]
    ),
})

print(diff_df.shape)
print(diff_df.head(10))

(204, 7)
   game_time_s  kills_diff  deaths_diff  assists_diff  hero_damage_diff  \
0     15.03125           0            0             0                 0   
1     30.03125           0            0             0                 0   
2     45.03125           0            0             0                 0   
3     60.03125           0            0             0               -87   
4     75.03125           0            0             0               247   
5     90.03125           0            0             0               378   
6    105.03125           0            0             0               402   
7    120.03125           0            0             0              -326   
8    135.03125           1           -1             1               154   
9    150.03125           1           -1             1              -381   

   objective_damage_diff  net_worth_diff  
0                      0           -1800  
1                      0           -1200  
2                      0            

In [16]:
# replay event lists into DataFrames
deaths_df = pd.DataFrame(replay_data["deaths"])
structures_df = pd.DataFrame(replay_data["structures_destroyed"])
mid_boss_df = pd.DataFrame(replay_data["mid_boss_kills"])

print("Deaths:")
print(deaths_df.head())
print("\nStructures:")
print(structures_df.head())
print("\nMid boss:")
print(mid_boss_df.head())

Deaths:
   team  player_index  game_time_s  death_duration_s
0     3            94   128.187500          8.062500
1     2            89   151.484375          8.062500
2     3            94   214.687500          8.046875
3     2            92   323.109375          8.062500
4     3            94   324.718750          8.062500

Structures:
   structure_type  team  destroyed_time_s
0  Tier2Structure     2        670.609375
1  Tier2Structure     3        784.265625
2  Tier2Structure     3        869.343750
3  Tier1Structure     2        903.500000
4  Tier2Structure     2        934.390625

Mid boss:
   killed_time_s
0    1598.609375
1    2563.921875
2    2954.109375


In [18]:
team_mapping = {
    2: "Team0",
    3: "Team1"
}

replay_team_features = team_snapshots.copy()

replay_team_features["perspective_team"] = (
    replay_team_features["team"].map(team_mapping)
)

print(replay_team_features[
    ["game_time_s", "team", "perspective_team"]
].head(20))

print("\nTeam mapping:")
print(replay_team_features["perspective_team"].value_counts())

    game_time_s  team perspective_team
0      15.03125     2            Team0
1      15.03125     3            Team1
2      30.03125     2            Team0
3      30.03125     3            Team1
4      45.03125     2            Team0
5      45.03125     3            Team1
6      60.03125     2            Team0
7      60.03125     3            Team1
8      75.03125     2            Team0
9      75.03125     3            Team1
10     90.03125     2            Team0
11     90.03125     3            Team1
12    105.03125     2            Team0
13    105.03125     3            Team1
14    120.03125     2            Team0
15    120.03125     3            Team1
16    135.03125     2            Team0
17    135.03125     3            Team1
18    150.03125     2            Team0
19    150.03125     3            Team1

Team mapping:
perspective_team
Team0    204
Team1    204
Name: count, dtype: int64


In [ ]:
# missing the team who claimed mid boss ???!!